# Справка

Убрать:
- __"Статус заказа"__ == "Заказ отменен до обработки" (ни на что не влияет, мусор)
- __"Дата оформления"__ < min("Дата отчета") - 60 дней
- __"Дата оформления"__ > max("Дата отчета")
- __"Ваш номер заказа"__ == *NaN*

In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
import pandas as pd
import numpy as np
import re

## Блок переменных, чтобы в одном месте скорректировать

+ **file** - здесь хранится имя файла для анализа

In [3]:
# file = 'united_orders_21570113_01-01-2025_15-01-2025.xlsx'
# file = 'united-orders-7ee29ed7-2669-45f2-b973-c7de37d0a59e (1).xlsx'
file = 'united_orders_21570113_01-01-2025_31-01-2025.xlsx'

## Блок загрузки файла и проверки соответствия имен

#### Листы

In [4]:
sheets = ['Сводка', 'Услуги и маржа по заказам', 'Транзакции по заказам и товарам']

missing_sheets = set(sheets) - set(pd.ExcelFile(file).sheet_names)
if missing_sheets:
    print(f'НЕ ХВАТАЕТ ЛИСТОВ: {missing_sheets}')
    print(pd.ExcelFile(file).sheet_names)

#### Проверка листа "Услуги и маржа по заказам" и получение df 'financials'

In [5]:
financials = pd.read_excel(file, sheet_name=sheets[1], header=6)
fin_cols = {
    'Номер заказа': 'Номер заказа'
    ,'Цена продажи (за шт.), ₽': 'Цена продажи (весь чек), ₽'
    ,'Все услуги Маркета за заказы, ₽': 'Все услуги Маркета за заказы, ₽'
    ,'Доход за вычетом услуг Маркета, ₽': 'Доход за вычетом услуг Маркета, ₽'
    ,'Статус заказа': 'Статус заказа'
    # , 'Статус платежа покупателя'
    # , 'Размещение товаров на витрине, ₽'
    # , 'Буст продаж, ₽'
    # , 'Доставка покупателю, ₽'
    # , 'Приём платежа покупателя, ₽'
    # , 'Перевод платежа покупателя, ₽'
    # , 'Обработка заказа, ₽'
}

missing_fin_cols = fin_cols.keys() - set(financials.columns)
if missing_fin_cols:
    print(f'НЕ ХВАТАЕТ КОЛОНОК: {missing_fin_cols}')
    print(financials.columns)

if not('Заказ отменен до обработки' in set(financials['Статус заказа'].unique())):
    financials['Статус заказа'] = financials['Статус заказа'].replace(
        to_replace='Заказ отменен до обработки' # Новое формулировка статуса
        ,value='Заказ отменен до обработки' # как настроено в данном скрипте
    )
    print(financials['Статус заказа'].unique())

#### Проверка листа "Транзакции по заказам и товарам" и получаем df 'transactions'

In [6]:
transactions = pd.read_excel(file, sheet_name=sheets[2], header=8)
trans_cols = {
        'Номер заказа': 'Номер заказа'
        ,'Дата оформления': 'Дата оформления'
        ,'Ваш SKU': 'Ваш SKU'
        ,'Название товара': 'Название товара'
        ,'Доставлено или возвращено': 'Количество'
        ,'Статус товара': 'Статус товара'
        ,'Цена продажи (за шт.), ₽': 'Цена продажи (за шт.), ₽'
}

goods_status_dict = {
    'Доставлен покупателю': 'Доставлен'
    ,'Отменён': 'Отменён'
    ,'Невыкуп принят на складе': 'Отменён'
    ,'Отгружен': 'В работе'
    ,'Невыкуп отправлен': 'Отменён'
    ,'Возврат принят на складе': 'Отменён'
    ,'Невыкуп принят': 'Отменён'
    ,'Невыкуп передан вам': 'Отменён'
    ,'Оформлен': 'В работе'
    ,'Возврат готов к передаче вам': 'Отменён'
    ,'Возврат передан вам': 'Отменён'
    ,'Невыкуп готов к передаче вам': 'Отменён'
    ,np.nan: np.nan
}

missing_trans_cols = trans_cols.keys() - set(transactions.columns)
if missing_trans_cols:
    print(f'НЕ ХВАТАЕТ КОЛОНОК: {missing_trans_cols}', end='\n\n')
    print(transactions.columns)

missing_goods_status = set(transactions['Статус товара'].unique()) - goods_status_dict.keys()
if missing_goods_status:
    print(f'НЕ ХВАТАЕТ СТАТУСА: {missing_goods_status}', end='\n\n')
    print(transactions['Статус товара'].unique())

НЕ ХВАТАЕТ СТАТУСА: {'Невыкуп утерян', 'Возврат отменён', 'Возврат оформлен'}

[nan 'Невыкуп принят на складе' 'Доставлен покупателю' 'Отменён'
 'Возврат оформлен' 'Возврат принят на складе' 'Невыкуп передан вам'
 'Возврат отменён' 'Возврат передан вам' 'Невыкуп утерян']


In [7]:
# Получим даты отчета, чтобы отфильтровать артефакты
summary = pd.read_excel(file,
                   sheet_name=sheets[0],
                   nrows=1,
                   header=None)
dates = re.findall(r"\d{2}\.\d{2}\.\d{4}", str(summary.iloc[0].values[0]))
dates_dt = pd.to_datetime(dates, format='%d.%m.%Y')

In [8]:
# оставляем необходимые для работы колонки и переименовываем
transactions = transactions[list(trans_cols.keys())].rename(columns=trans_cols)
financials = financials[list(fin_cols.keys())].rename(columns=fin_cols).fillna(0.0)
# Заполняем NaN нулями, чтобы мат. функции нормально отрабатывали

In [9]:
transactions = transactions.dropna(subset=['Номер заказа']) # заказ без номера

In [10]:
# преобразование форматов
transactions['Дата оформления'] = pd.to_datetime(
    transactions['Дата оформления'],
    format='%d.%m.%Y',
    errors='coerce' # NaT, если формат не совпадает
)
transactions = transactions.copy()
financials = financials.copy()
transactions['Количество'] = transactions['Количество'].fillna(0)
financials['Статус заказа'] = financials['Статус заказа'].astype('category')

In [11]:
# Преобразовать в целое число
to_int = ['Номер заказа'
          ,'Количество']

transactions[to_int] = transactions[to_int].astype('int64')

In [12]:
# Соединим транзации и расходы на услуги Маркета
transactions = transactions.merge(financials, on='Номер заказа', how='left')

In [13]:
# Удаляем записи со статусом 'Заказ отменен до обработки', так как в них нет смысла для бизнеса
transactions = transactions[transactions['Статус заказа'] != 'Заказ отменен до обработки']

In [14]:
# сокращаем статусы товаров до 3 - "доставлен", "отменен", "в работе"
transactions['Статус товара'] = (
    transactions['Статус товара']
    .replace(goods_status_dict)
    .astype('category')
)

In [15]:
# Добавим колонку "Стоимость товаров, ₽" = "Количество" * "Цена продажи, ₽"
transactions['Стоимость товаров, ₽'] = transactions['Количество'].mul(transactions['Цена продажи (за шт.), ₽'])

In [16]:
# Коэффициент от чека (доля стоимости товаров от продажной цены)
transactions['Коэффициент от чека'] = transactions['Стоимость товаров, ₽'] / transactions['Цена продажи (весь чек), ₽']

In [17]:
# пересчитываем колонки с учетом коэффициента доли в чеке
coeff_cols = ['Все услуги Маркета за заказы, ₽', 'Доход за вычетом услуг Маркета, ₽']

def apply_coefficient(df, cols, coeff_col, round_decimals=2):
    df = df.copy()
    df[cols] = df[cols].multiply(df[coeff_col].values, axis=0).round(round_decimals)
    return df

transactions = apply_coefficient(transactions, coeff_cols, 'Коэффициент от чека')


In [18]:
col_for_result = ['Ваш SKU', 'Количество', 'Статус товара', 'Цена продажи (за шт.), ₽', 'Все услуги Маркета за заказы, ₽',
                  'Доход за вычетом услуг Маркета, ₽']

final_df = transactions[transactions['Статус товара'] != 'В работе'][col_for_result]

In [19]:
cost_goods = final_df.groupby('Ваш SKU')['Цена продажи (за шт.), ₽'].agg(
    Минимум='min',
    Максимум='max',
    Среднее='mean'
).round(2)

In [20]:
sku_income = final_df[final_df['Статус товара'] == 'Доставлен'].groupby('Ваш SKU')['Доход за вычетом услуг Маркета, ₽'].sum()

In [21]:
result_div_and_cancel = final_df.groupby('Ваш SKU').agg(
    Услуги_Маркета=('Все услуги Маркета за заказы, ₽', 'sum'),
    sum_div_and_cancel=('Количество', 'sum')
)

In [22]:
cancelled_counts = final_df[
    final_df['Статус товара'] == 'Отменён'
].groupby('Ваш SKU').agg(
    Количество_отменённых=('Количество', 'sum')
)

- cost_goods - df с ценами (min, max, mean)
- sku_income - df с доходом за вычетом услуг Маркета (только Доставленные товары)
- result_div_and_cancel - df Расходы на Маркет и Кол-во товаров (доставлены и отмененные)
- cancelled_counts - df с кол-вом штук товаров со статусом отмена

In [23]:
merged_df = cost_goods.merge(sku_income, how='left', on='Ваш SKU')
merged_df = merged_df.merge(result_div_and_cancel, how='left', on='Ваш SKU')
merged_df = merged_df.merge(cancelled_counts, how='left', on='Ваш SKU')
merged_df = merged_df.fillna(0)

In [24]:
merged_df['Процент_отменённых, %'] = (merged_df['Количество_отменённых'] / merged_df['sum_div_and_cancel']).round(4) * 100
merged_df['Количество доставленных, шт'] = merged_df['sum_div_and_cancel'] - merged_df['Количество_отменённых']

In [25]:
name_goods = transactions.drop_duplicates(subset=['Ваш SKU'], keep='last')[['Ваш SKU', 'Название товара']].set_index('Ваш SKU')
merged_df = merged_df.merge(name_goods, how='left', on='Ваш SKU')

In [26]:
merged_df = merged_df.rename(columns={
    'Минимум': 'Мин. цена, ₽',
    'Максимум': 'Макс. цена, ₽',
    'Среднее': 'Средняя цена, ₽',
    'Доход за вычетом услуг Маркета, ₽': 'Чистый доход, ₽',
    'Услуги_Маркета': 'Комиссия Маркета, ₽',
    'Процент_отменённых, %': 'Доля отмен, %',
    'Количество доставленных, шт': 'Доставлено, шт',
    'Название товара': 'Наименование товара'
})

orderliness = [
    'Наименование товара',
    'Мин. цена, ₽',
    'Макс. цена, ₽',
    'Средняя цена, ₽',
    'Доставлено, шт',
    'Чистый доход, ₽',
    'Комиссия Маркета, ₽',
    'Доля отмен, %'
]

merged_df = merged_df[orderliness].sort_values(by='Чистый доход, ₽', ascending=False)

In [27]:
summary_df = transactions.groupby('Статус товара', observed=True)[['Доход за вычетом услуг Маркета, ₽', 'Все услуги Маркета за заказы, ₽']].sum()

In [28]:
with pd.ExcelWriter(file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    summary_df.to_excel(writer, sheet_name='Сводка по Статусам', index=True)
    merged_df.to_excel(writer, sheet_name='Сводка по SKU', index=True)